In [1]:
from DatasetLoading import RepairDatasetLoader
from RayRepresentationEncoding import RayRepresentationEncoder
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import lightning as L
import torch
from lightning.pytorch.loggers import WandbLogger
from scipy.spatial.transform import Rotation


In [2]:
torch.set_float32_matmul_precision('medium')

In [3]:
#test_load = np.load(dataset_loader.test_ray_files[0])
#test_ray_colours = torch.from_numpy(test_load['ray_colours'])
#test_piece_rotation = torch.from_numpy(test_load['pi ece_rotation'])
#test_rays = torch.from_numpy(test_load['rays'])


In [4]:
class SimpleRotationPredictionNetwork(nn.Module):
    def __init__(self):
        super().__init__()


        representation_size = 128 * 2

        self.representation_encoder = RayRepresentationEncoder(transformer_layers=1, representation_size=representation_size//2, initial_dropout=0)

        transformer_layer = nn.TransformerEncoderLayer(d_model=representation_size, nhead=8, dropout=0, batch_first=True)
        self.transformer = nn.TransformerEncoder(transformer_layer, num_layers=1)


        self.head = nn.Sequential(nn.Linear(representation_size, 64),
                                  nn.BatchNorm1d(64),
                                  nn.ReLU(),
                                  nn.Linear(64, 4))

        self.pos_encoding = nn.Parameter(torch.randn(5000, representation_size), requires_grad=True)



    def forward(self, ray_colours_1, ray_locations_1, ray_colours_2, ray_locations_2):
        del ray_locations_1, ray_locations_2

        x = torch.cat((ray_colours_1, ray_colours_2), dim=1)
        batch_size = x.size(0)
        num_rays = ray_colours_1.size(1)
        del ray_colours_1, ray_colours_2


        x = x.view(batch_size * 2, num_rays, -1)


        x = self.representation_encoder(x)


        x = x.view(batch_size, num_rays * 2, -1)

        x = torch.cat([x[:, num_rays:, :], x[:, :num_rays, :]], dim=2)

        x = x + self.pos_encoding

        x = self.transformer(x)


        x = torch.mean(x, dim=1)

        #x = x.view(batch_size, 2 * x.size(1))

        x = self.head(x)

        return x

class PairSimpleRotationPrediction(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = SimpleRotationPredictionNetwork()
        self.lr = 1e-4

    def calculate_loss(self, batch, batch_idx, stage):
        #ray_colours_1, ray_locations_1, ray_colours_2, ray_locations_2, gt_rotation, rot_num = batch
        #onehot_rot_num = torch.nn.functional.one_hot(rot_num, num_classes=4).float()
        #predicted_rotation = self.model(ray_colours_1, ray_locations_1, ray_colours_2, ray_locations_2)
        #try:

            #loss = nn.functional.cross_entropy(predicted_rotation, onehot_rot_num)
            #self.log(stage + '_cross_entropy_loss', loss)

            #preds = predicted_rotation.argmax(dim=-1)
            #acc = (preds == rot_num).float().mean()
            #self.log(stage + '_accuracy', acc)
        #except Exception as e:
         #   print("Error in loss calculation: ", e)
          #  print(predicted_rotation.shape, onehot_rot_num.shape)

        ray_colours_1, ray_locations_1, ray_colours_2, ray_locations_2, gt_rotation = batch
        predicted_rotation = self.model(ray_colours_1, ray_locations_1, ray_colours_2, ray_locations_2)
        loss = nn.functional.mse_loss(predicted_rotation, gt_rotation)
        self.log(stage + '_loss', loss)


        cos_theta = torch.sum(predicted_rotation * gt_rotation, dim=-1)
        cos_theta = torch.clamp(cos_theta, -1.0, 1.0)
        rot_rmse = torch.acos(cos_theta)
        rot_rmse = torch.rad2deg(rot_rmse)
        rot_rmse = torch.sqrt(rot_rmse.pow(2).mean())
        self.log(stage + '_angular_error', rot_rmse)

        del ray_colours_1, ray_locations_1, ray_colours_2, ray_locations_2, gt_rotation, predicted_rotation, cos_theta
        return rot_rmse

    def training_step(self, batch, batch_idx):

        return self.calculate_loss(batch, batch_idx, stage='train')

    def test_step(self, batch, batch_idx):
        self.calculate_loss(batch, batch_idx, stage='test')

    def validation_step(self, batch, batch_idx):
        self.calculate_loss(batch, batch_idx, stage='val')

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        return optimizer


In [5]:
wandb_logger = WandbLogger()

In [6]:
dataset_loader = RepairDatasetLoader(batch_size=12, dataset_type="RotatedRayPairsDataset", rays_folder_name="RaysSRot")

In [ ]:
L.seed_everything(42)
model = PairSimpleRotationPrediction()
trainer = L.Trainer(max_epochs=50, logger=wandb_logger, accelerator='gpu', accumulate_grad_batches=5)
trainer.fit(model, datamodule=dataset_loader)

Seed set to 42
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/enego/.netrc.
wandb: Currently logged in as: enegocomley (enegocomley-enego-couk) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------
0 | model | SimpleRotationPredictionNetwork | 4.1 M  | train | 0    
--------------------------------------------------------------------------
4.1 M     Trainable params
0         Non-trainable params
4.1 M     Total params
16.539    Total estimated model params size (MB)
39        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/enego/Documents/masters/rayRepresntationTesting/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/enego/Documents/masters/rayRepresntationTesting/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:485: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


Epoch 0:   0%|          | 11/2310 [00:08<30:33,  1.25it/s, v_num=nnzr]     